In [ ]:
!pip install -q streamlit streamlit-option-menu pyngrok pyjwt bcrypt plotly


In [ ]:
%%writefile otp_service.py

import os
import secrets
import smtplib
import datetime
import time
import jwt
import bcrypt
from email.utils import formatdate, make_msgid
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

# CRYPTO & HASHING HELPERS

def hash_txt(text: str) -> str:

    if not text:
        return ""
    return bcrypt.hashpw(text.encode("utf-8"), bcrypt.gensalt()).decode("utf-8")

def check_txt(text: str, hashed: str) -> bool:

    if not text or not hashed:
        return False
    try:
        return bcrypt.checkpw(text.encode("utf-8"), hashed.encode("utf-8"))
    except Exception:
        return False
# CRYPTOGRAPHICALLY SECURE OTP GENERATION

def generate_otp() -> str:

    random_int = secrets.randbelow(1000000)
    return f"{random_int:06d}"

# JWT OTP TOKEN CREATION & VERIFICATION

def make_otp_token(email: str, otp: str, jwt_secret: str, expiry_minutes: int = 5) -> str:

    if not jwt_secret:
        raise ValueError("JWT signing secret must be provided.")

    now = datetime.datetime.now(datetime.timezone.utc)
    payload = {
        "sub": email.lower().strip(),
        "otp_hash": hash_txt(otp),
        "type": "password_reset_otp",
        "jti": secrets.token_hex(16),
        "iat": now,
        "exp": now + datetime.timedelta(minutes=expiry_minutes)
    }
    return jwt.encode(payload, jwt_secret, algorithm="HS256")


def verify_otp_token(
    token: str,
    input_otp: str,
    email: str,
    jwt_secret: str,
    expiry_minutes: int = 5
) -> tuple:

    if not token:
        return False, "Verification token missing. Please request a new OTP."

    cleaned_otp = input_otp.strip() if input_otp else ""
    if len(cleaned_otp) != 6 or not cleaned_otp.isdigit():
        return False, "Please enter a valid 6-digit numerical OTP code."

    try:
        payload = jwt.decode(token, jwt_secret, algorithms=["HS256"])

        # Validate subject email
        if payload.get("sub") != email.lower().strip():
            return False, "Security verification failed: Token does not match the active account."

        # Validate token purpose
        if payload.get("type") != "password_reset_otp":
            return False, "Invalid token type for password recovery."

        # Verify OTP hash
        stored_otp_hash = payload.get("otp_hash")
        if check_txt(cleaned_otp, stored_otp_hash):
            return True, "Valid"

        return False, "Invalid or incorrect 6-digit OTP code. Please check and try again."

    except jwt.ExpiredSignatureError:
        return False, f"⚠️This OTP code expired after {expiry_minutes} minutes. Please request a new code."
    except jwt.InvalidSignatureError:
        return False, "Security verification failed: Tampered or invalid token signature."
    except Exception:
        return False, "Invalid or corrupted verification session."


# RATE LIMITING & COOLDOWN HELPERS

def can_request_otp(last_sent_timestamp: float, cooldown_seconds: int = 60) -> tuple:

    if not last_sent_timestamp:
        return True, 0

    elapsed = time.time() - last_sent_timestamp
    if elapsed >= cooldown_seconds:
        return True, 0

    remaining = int(cooldown_seconds - elapsed) + 1
    return False, remaining

# PASSWORD HISTORY & REUSE PREVENTION (SIR'S CODE INTEGRATION)

def record_password_history(conn, email: str, password_hash: str) -> None:

    now_str = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
    conn.execute(
        "INSERT INTO password_history (email, password_hash, set_at) VALUES (?, ?, ?)",
        (email.lower().strip(), password_hash, now_str)
    )
    conn.commit()


def check_password_reused(conn, email: str, new_password: str, history_limit: int = 5) -> bool:

    cursor = conn.cursor()
    cursor.execute(
        "SELECT password_hash FROM password_history WHERE email = ? ORDER BY id DESC LIMIT ?",
        (email.lower().strip(), history_limit)
    )
    rows = cursor.fetchall()
    for (old_hash,) in rows:
        if check_txt(new_password, old_hash):
            return True
    return False

# GMAIL SMTP MULTIPART EMAIL DELIVERY

def send_professional_email(
    sender_email: str,
    to_email: str,
    otp: str,
    app_pass: str,
    expiry_minutes: int = 5
) -> tuple:

    if not app_pass:
        return False, "Gmail App Password not found! Please configure EMAIL_PASSWORD in Colab Secrets."
    if not sender_email:
        return False, "Sender email address not configured. Please configure SENDER_EMAIL."

    msg = MIMEMultipart("alternative")
    msg["From"] = f"FreightQuote AI Verification <{sender_email}>"
    msg["To"] = to_email
    msg["Subject"] = "FreightQuote AI Verification - Password Reset Code"
    msg["Date"] = formatdate(localtime=True)
    msg["Message-ID"] = make_msgid()
    msg["Reply-To"] = sender_email

    # Plain-text version
    text_body = (
        f"FreightQuote AI Verification\n\n"
        f"We received a request to reset your password for {to_email}.\n\n"
        f"Your 6-Digit Verification Code is: {otp}\n\n"
        f"This code will expire in {expiry_minutes} minutes.\n"
        f"If you did not request this password reset, please ignore this email.\n\n"
        f"© 2026 FreightQuote AI. All rights reserved."
    )


    html_body = f"""<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <style>
        body {{
            font-family: 'Segoe UI', Arial, sans-serif;
            background-color: #F5FAFF;
            margin: 0;
            padding: 24px;
            color: #475569;
        }}
        .container {{
            max-width: 520px;
            margin: 0 auto;
            background-color: #FFFFFF;
            border: 1px solid #E2E8F0;
            border-radius: 12px;
            padding: 32px 28px;
            text-align: center;
        }}
        .header {{
            font-size: 24px;
            font-weight: 700;
            color: #475569;
            margin-bottom: 12px;
        }}
        .subtext {{
            font-size: 14px;
            color: #64748B;
            line-height: 1.6;
            margin-bottom: 24px;
        }}
        .otp-container {{
            margin: 20px 0;
        }}
        .otp-box {{
            background-color: #E4F2FF;
            color: #475569;
            font-size: 32px;
            font-weight: 700;
            letter-spacing: 8px;
            padding: 14px 28px;
            border: 1px solid #8CCBFA;
            border-radius: 8px;
            display: inline-block;
        }}
        .timer {{
            font-size: 13px;
            color: #64748B;
            font-weight: 500;
            margin-top: 14px;
        }}
        .footer {{
            font-size: 12px;
            color: #94A3B8;
            margin-top: 28px;
            border-top: 1px solid #E2E8F0;
            padding-top: 16px;
            line-height: 1.5;
        }}
    </style>
</head>
<body>
    <div class="container">
       <div class="header">FreightQuote AI Verification</div>
        <div class="subtext">
            We received a request to reset the password for <b>{to_email}</b>.<br>
            Please use the 6-digit verification code below:
        </div>
        <div class="otp-container">
            <div class="otp-box">{otp}</div>
            <div class="timer">Valid for <b>{expiry_minutes} minutes</b></div>
        </div>
        <div class="footer">
            If you did not initiate this request, you can safely disregard this email.<br>
            &copy; 2026 FreightQuote AI. All rights reserved.
        </div>
    </div>
</body>
</html>"""

    msg.attach(MIMEText(text_body, "plain"))
    msg.attach(MIMEText(html_body, "html"))

    server = None
    try:
        server = smtplib.SMTP("smtp.gmail.com", 587, timeout=15)
        server.starttls()
        server.login(sender_email, app_pass)
        server.sendmail(sender_email, to_email, msg.as_string())
        return True, "Email sent successfully!"
    except smtplib.SMTPAuthenticationError:
        return False, "SMTP Authentication Failed: Please verify your Gmail App Password."
    except Exception as e:
        return False, f"Failed to send email: {str(e)}"
    finally:
        if server:
            try:
                server.quit()
            except Exception:
                pass


In [ ]:
%%writefile app.py
import os, sqlite3, jwt, bcrypt, datetime, time, re, subprocess
import streamlit as st
from streamlit_option_menu import option_menu

#Import our comprehensive OTP service from otp_service.py
from otp_service import (
    hash_txt, check_txt, generate_otp, make_otp_token,
    verify_otp_token, send_professional_email,
    can_request_otp, record_password_history, check_password_reused
)


#FORCE STREAMLIT LIGHT THEME CONFIGURATION

os.makedirs(".streamlit", exist_ok=True)
with open(".streamlit/config.toml", "w") as f:
    f.write(
        '[theme]\n'
        'base="light"\n'
        'primaryColor="#8CCBFA"\n'
        'backgroundColor="#F5FAFF"\n'
        'secondaryBackgroundColor="#FFFFFF"\n'
        'textColor="#475569"\n'
    )

st.set_page_config(
    page_title="FreightQuote AI",
    layout="wide",
    initial_sidebar_state="expanded"
)

COLORS = {
    "bg_main": "#F5FAFF",
    "bg_sidebar": "#FFFFFF",
    "bg_card": "#FFFFFF",
    "bg_card_alt": "#E4F2FF",
    "text_main": "#475569",
    "text_heading": "#475569",
    "text_muted": "#64748B",
    "accent": "#8CCBFA",
    "accent_hover": "#E4F2FF",
    "accent_active": "#72BAF0",
    "accent_text": "#475569",
    "border": "#E2E8F0",
    "border_light": "#E2E8F0",
    "disabled": "#F1F5F9",
    "success": "#34D399",
    "danger": "#F87171"
}

# SECURE CONFIGURATION RETRIEVAl

def get_secret(key, default=None):

    val = os.getenv(key)
    if val:
        return val
    try:
        from google.colab import userdata
        val = userdata.get(key)
        if val:
            return val
    except Exception:
        pass
    return default

JWT_SECRET = get_secret("JWT_SECRET", "infosys-portal-jwt-secret-key-2026-min32bytes")
SENDER_EMAIL = get_secret("EMAIL_ID") or get_secret("SENDER_EMAIL", "tirth12222@gmail.com")
EMAIL_PASSWORD = get_secret("EMAIL_PASSWORD", None)
ADMIN_EMAIL = get_secret("ADMIN_EMAIL", "infosys@ai")
ADMIN_PASSWORD = get_secret("ADMIN_PASSWORD", "Admin@123")
OTP_EXPIRY_MINUTES = 5

st.markdown(f"""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap');

    html, body, .stApp {{
        background: {COLORS['bg_main']} !important;
        font-family: 'Inter', -apple-system, BlinkMacSystemFont, sans-serif !important;
        color: {COLORS['text_main']} !important;
    }}

    footer, div[data-testid="stDecoration"] {{ visibility: hidden !important; display: none !important; }}
    header {{ background: transparent !important; z-index: 999999 !important; }}

    /* Sidebar toggle button */
    button[kind="header"], div[data-testid="stSidebarCollapsedControl"] button {{
        visibility: visible !important;
        display: flex !important;
        opacity: 1 !important;
        background-color: {COLORS['accent']} !important;
        border: 1px solid {COLORS['border']} !important;
        border-radius: 6px !important;
        padding: 6px !important;
        margin: 8px !important;
    }}
    button[kind="header"] svg, div[data-testid="stSidebarCollapsedControl"] svg {{
        fill: {COLORS['text_heading']} !important;
        color: {COLORS['text_heading']} !important;
        stroke: {COLORS['text_heading']} !important;
    }}

    .block-container {{ padding: 2rem 2.5rem !important; max-width: 1080px; }}
    h1, h2, h3, h4 {{
        font-family: 'Inter', sans-serif !important;
        color: {COLORS['text_heading']} !important;
        font-weight: 700 !important;
    }}
    label p {{
        font-weight: 500 !important;
        color: {COLORS['text_main']} !important;
        font-size: 14px !important;
    }}

    /* Input Fields */
    div[data-baseweb="base-input"] {{
        background-color: transparent !important;
        border: none !important;
    }}
    div[data-baseweb="input"] {{
        background-color: {COLORS['bg_card']} !important;
        border: 1px solid {COLORS['border']} !important;
        border-radius: 8px !important;
        transition: all 0.18s ease !important;
    }}
    div[data-baseweb="input"]:focus-within {{
        border-color: {COLORS['accent']} !important;
        box-shadow: 0 0 0 3px {COLORS['accent_hover']} !important;
    }}
    input {{
        color: {COLORS['text_main']} !important;
        -webkit-text-fill-color: {COLORS['text_main']} !important;
        font-size: 14px !important;
    }}

    /* Selectbox (Dropdown) Styling */
    div[data-baseweb="select"] > div {{
        background-color: {COLORS['bg_card']} !important;
        border: 1px solid {COLORS['border']} !important;
        border-radius: 8px !important;
        color: {COLORS['text_main']} !important;
        transition: all 0.18s ease !important;
    }}
    div[data-baseweb="select"]:focus-within > div {{
        border-color: {COLORS['accent']} !important;
        box-shadow: 0 0 0 3px {COLORS['accent_hover']} !important;
    }}
    div[data-baseweb="select"] span {{
        color: {COLORS['text_main']} !important;
    }}
    ul[role="listbox"] {{
        background-color: {COLORS['bg_card']} !important;
        border: 1px solid {COLORS['border']} !important;
    }}
    li[role="option"] {{
        color: {COLORS['text_main']} !important;
        background-color: {COLORS['bg_card']} !important;
    }}
    li[role="option"]:hover, li[role="option"][aria-selected="true"] {{
        background-color: {COLORS['accent_hover']} !important;
        color: {COLORS['text_heading']} !important;
    }}

    /* Buttons with Soft Sky-Blue Theme & Smooth Transitions */
    div[data-testid="stButton"] button {{
        background-color: {COLORS['accent']} !important;
        color: {COLORS['accent_text']} !important;
        border: 1px solid {COLORS['accent']} !important;
        border-radius: 8px !important;
        font-family: 'Inter', sans-serif !important;
        font-weight: 600 !important;
        font-size: 14px !important;
        height: 44px !important;
        min-height: 44px !important;
        white-space: nowrap !important;
        display: flex !important;
        align-items: center !important;
        justify-content: center !important;
        padding: 0px 16px !important;
        width: 100%;
        transition: all 0.18s ease !important;
        cursor: pointer !important;
        box-shadow: none !important;
    }}
    div[data-testid="stButton"] button:hover {{
        background-color: {COLORS['accent_hover']} !important;
        border-color: {COLORS['accent']} !important;
        color: {COLORS['text_heading']} !important;
    }}
    div[data-testid="stButton"] button:active {{
        background-color: {COLORS['accent_active']} !important;
        transform: translateY(1px) !important;
    }}
    div[data-testid="stButton"] button:disabled {{
        background-color: {COLORS['disabled']} !important;
        border-color: {COLORS['border']} !important;
        color: {COLORS['text_muted']} !important;
        cursor: not-allowed !important;
    }}

    /* Clean Card Layouts */
    section[data-testid="stSidebar"] {{
        background: {COLORS['bg_sidebar']} !important;
        border-right: 1px solid {COLORS['border']} !important;
    }}
    .pn-card {{
        background: {COLORS['bg_card']};
        border: 1px solid {COLORS['border']};
        border-radius: 10px;
        padding: 24px;
        box-shadow: none;
    }}
    .account-row {{
        display: flex;
        justify-content: space-between;
        padding: 10px 0;
        border-bottom: 1px solid {COLORS['border']};
        font-size: 14px;
    }}
    .account-label {{
        color: {COLORS['text_muted']};
        font-weight: 500;
    }}
    .account-value {{
        color: {COLORS['text_main']};
        font-weight: 600;
    }}
</style>
""", unsafe_allow_html=True)


# DATABASE INITIALIZATION & MIGRATION

def get_db():
    return sqlite3.connect("infosys_portal.db", check_same_thread=False)

with get_db() as conn:
    conn.execute("""CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        username TEXT UNIQUE,
        email TEXT UNIQUE,
        password_hash TEXT,
        security_question TEXT,
        security_answer_hash TEXT,
        role TEXT DEFAULT 'user',
        created_at TEXT)""")

    conn.execute("""CREATE TABLE IF NOT EXISTS password_history (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        email TEXT,
        password_hash TEXT,
        set_at TEXT,
        FOREIGN KEY(email) REFERENCES users(email))""")

    cols = [col[1] for col in conn.execute("PRAGMA table_info(users)").fetchall()]
    if "role" not in cols:
        conn.execute("ALTER TABLE users ADD COLUMN role TEXT DEFAULT 'user'")
    if "created_at" not in cols:
        conn.execute("ALTER TABLE users ADD COLUMN created_at TEXT")

    now_str = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
    admin_record = conn.execute("SELECT id FROM users WHERE email=? OR role='admin'", (ADMIN_EMAIL,)).fetchone()
    if not admin_record:
        admin_h = hash_txt(ADMIN_PASSWORD)
        conn.execute(
            "INSERT INTO users (username, email, password_hash, security_question, security_answer_hash, role, created_at) "
            "VALUES (?, ?, ?, ?, ?, ?, ?)",
            ("Administrator", ADMIN_EMAIL, admin_h, "What is your pet name?", hash_txt("admin"), "admin", now_str)
        )
        record_password_history(conn, ADMIN_EMAIL, admin_h)

# INPUT VALIDATION UTILITIES

def validate_email(email_str):

    if not email_str or "@" not in email_str:
        return False, "Email address must contain '@'."
    parts = email_str.split("@")
    if len(parts) != 2:
        return False, "Invalid email address format."
    local_part, domain_part = parts[0], parts[1]
    if len(local_part) < 2:
        return False, "Email must have at least 2 characters before the '@' symbol."
    if "." not in domain_part:
        return False, "Email domain must contain a dot (.)."
    domain_parts = domain_part.rsplit(".", 1)
    domain_name, top_domain = domain_parts[0], domain_parts[1]
    if len(domain_name) < 2:
        return False, "Email must have at least 2 characters between '@' and the domain dot."
    if len(top_domain) < 2:
        return False, "Email must have at least 2 characters after the domain dot."
    email_pattern = r"^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$"
    if not re.match(email_pattern, email_str):
        return False, "Email contains invalid characters."
    return True, "Valid"

def validate_password(pwd):

    if len(pwd) < 8:
        return False, "Password must be at least 8 characters long."
    if not any(c.isupper() for c in pwd):
        return False, "Password must include at least one uppercase letter (A-Z)."
    if not any(c.islower() for c in pwd):
        return False, "Password must include at least one lowercase letter (a-z)."
    if not any(c.isdigit() for c in pwd):
        return False, "Password must include at least one digit (0-9)."
    special_chars = set("!@#$%^&*()_+-=[]{}|;':,./<>?`~\"")
    if not any(c in special_chars for c in pwd):
        return False, "Password must include at least one special character (!@#$%^&*)."
    return True, "Valid"


# JWT SESSION MANAGEMENT

def make_jwt(email, role="user"):
    now = datetime.datetime.now(datetime.timezone.utc)
    payload = {
        "email": email,
        "role": role,
        "iat": now,
        "exp": now + datetime.timedelta(hours=2)
    }
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")

def verify_jwt(token):
    try:
        return jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except Exception:
        return None


# SESSION STATE INITIALIZATION & CLEANUP HELPERS

state_defaults = {
    "token": None,
    "page": "Login",
    "reset_email": None,
    "reset_mode": None,
    "sq_p": None,
    "otp_token": None,
    "otp_verified": False,
    "otp_stage": "email",
    "otp_attempts": 0,
    "otp_last_sent_at": 0
}
for key, val in state_defaults.items():
    if key not in st.session_state:
        st.session_state[key] = val

def navigate(p):
    st.session_state.page = p
    st.rerun()

def clear_reset_state():
    st.session_state.reset_email = None
    st.session_state.reset_mode = None
    st.session_state.sq_p = None
    st.session_state.otp_token = None
    st.session_state.otp_verified = False
    st.session_state.otp_stage = "email"
    st.session_state.otp_attempts = 0

def auth_header(title, sub="Intelligent Authentication Portal"):
    st.markdown(f"""
    <div style="width:100%;text-align:center;padding:1.5rem 0 1rem;box-sizing:border-box;">
        <h1 style="font-size:1.85rem !important;margin:0;color:{COLORS['text_heading']};font-weight:700;">FreightQuote AI</h1>
        <p style="color:{COLORS['text_muted']};font-size:14px;margin:4px 0 0;">{sub}</p>
    </div>
    <div style="text-align:center;margin-bottom:1.5rem;">
        <span style="font-size:1.05rem;font-weight:600;color:{COLORS['text_main']};">{title}</span>
    </div>
    """, unsafe_allow_html=True)


# PAGE ROUTING

if not st.session_state.token:
    if st.session_state.page not in ["Login", "Signup", "Forgot"]:
        st.session_state.page = "Login"

    _, mid, _ = st.columns([1, 1.45, 1])
    with mid:
        # LOGIN PAGE

        if st.session_state.page == "Login":
            auth_header("Sign in to your account")
            login_id = st.text_input("Username or Email", placeholder="Enter your username or email").strip()
            pwd = st.text_input("Password", type="password", placeholder="Enter your password")
            st.markdown("<br>", unsafe_allow_html=True)

            col_l, col_c, col_r = st.columns([1, 1.15, 1.3])
            if col_l.button("Sign In", use_container_width=True):
                if not login_id or not pwd:
                    st.error("Both username/email and password fields are required.")
                else:
                    with get_db() as c:
                        r = c.execute(
                            "SELECT email, password_hash, role FROM users WHERE email=? OR username=?",
                            (login_id.lower(), login_id)
                        ).fetchone()
                    if r and check_txt(pwd, r[1]):
                        user_email = r[0]
                        user_role = r[2] if len(r) > 2 and r[2] else "user"
                        st.session_state.token = make_jwt(user_email, user_role)
                        navigate("Dashboard")
                    else:
                        st.error("Invalid credentials. Please check your username/email and password.")

            if col_c.button("Create Account", use_container_width=True):
                st.session_state.signup_errors = {}
                navigate("Signup")
            if col_r.button("Forgot Password", use_container_width=True):
                clear_reset_state()
                navigate("Forgot")


        # SIGNUP PAGE

        elif st.session_state.page == "Signup":
            auth_header("Create an account", "Your intelligent maritime freight platform")
            st.markdown(f"""
            <div style="font-size:13px; color:{COLORS['text_muted']}; margin-top:-6px; margin-bottom:14px;">
                <span style="color:#e11d48; font-weight:600;">*</span> All fields marked with an asterisk are required.
            </div>
            """, unsafe_allow_html=True)

            if "signup_errors" not in st.session_state:
                st.session_state.signup_errors = {}
            signup_errs = st.session_state.signup_errors

            # Subtle error border styling for missing fields
            error_styles = []
            if "uname" in signup_errs:
                error_styles.append('div[data-testid="stTextInput"]:has(input[aria-label="Username *"]) div[data-baseweb="input"] { border: 1.5px solid #f87171 !important; background-color: #fffbfa !important; }')
            if "email" in signup_errs:
                error_styles.append('div[data-testid="stTextInput"]:has(input[aria-label="Email address *"]) div[data-baseweb="input"] { border: 1.5px solid #f87171 !important; background-color: #fffbfa !important; }')
            if "pwd" in signup_errs:
                error_styles.append('div[data-testid="stTextInput"]:has(input[aria-label="Password *"]) div[data-baseweb="input"] { border: 1.5px solid #f87171 !important; background-color: #fffbfa !important; }')
            if "confirm_pwd" in signup_errs:
                error_styles.append('div[data-testid="stTextInput"]:has(input[aria-label="Confirm password *"]) div[data-baseweb="input"] { border: 1.5px solid #f87171 !important; background-color: #fffbfa !important; }')
            if "sq" in signup_errs:
                error_styles.append('div[data-testid="stSelectbox"] div[data-baseweb="select"] > div { border: 1.5px solid #f87171 !important; background-color: #fffbfa !important; }')
            if "sa" in signup_errs:
                error_styles.append('div[data-testid="stTextInput"]:has(input[aria-label="Security answer *"]) div[data-baseweb="input"] { border: 1.5px solid #f87171 !important; background-color: #fffbfa !important; }')

            if error_styles:
                st.markdown(f"<style>{' '.join(error_styles)}</style>", unsafe_allow_html=True)

            uname = st.text_input("Username *", key="signup_uname", placeholder="e.g. janedoe").strip()
            if "uname" in signup_errs:
                st.markdown(f'<div style="color:#e11d48;font-size:12px;margin-top:-8px;margin-bottom:10px;font-weight:500;">{signup_errs["uname"]}</div>', unsafe_allow_html=True)

            email = st.text_input("Email address *", key="signup_email", placeholder="e.g. ab@cd.ef").lower().strip()
            if "email" in signup_errs:
                st.markdown(f'<div style="color:#e11d48;font-size:12px;margin-top:-8px;margin-bottom:10px;font-weight:500;">{signup_errs["email"]}</div>', unsafe_allow_html=True)

            pwd = st.text_input("Password *", key="signup_pwd", type="password", placeholder="Min. 8 chars (upper, lower, digit, symbol)")
            if "pwd" in signup_errs:
                st.markdown(f'<div style="color:#e11d48;font-size:12px;margin-top:-8px;margin-bottom:10px;font-weight:500;">{signup_errs["pwd"]}</div>', unsafe_allow_html=True)

            confirm_pwd = st.text_input("Confirm password *", key="signup_confirm_pwd", type="password", placeholder="Re-enter password")
            if "confirm_pwd" in signup_errs:
                st.markdown(f'<div style="color:#e11d48;font-size:12px;margin-top:-8px;margin-bottom:10px;font-weight:500;">{signup_errs["confirm_pwd"]}</div>', unsafe_allow_html=True)

            sq_options = [
                "Select a security question...",
                "What is your pet name?",
                "What is your mother's maiden name?",
                "What is your favourite city?"
            ]
            sq = st.selectbox("Security Question *", sq_options, key="signup_sq")
            if "sq" in signup_errs:
                st.markdown(f'<div style="color:#e11d48;font-size:12px;margin-top:-8px;margin-bottom:10px;font-weight:500;">{signup_errs["sq"]}</div>', unsafe_allow_html=True)

            sa = st.text_input("Security answer *", key="signup_sa", placeholder="Your answer").strip()
            if "sa" in signup_errs:
                st.markdown(f'<div style="color:#e11d48;font-size:12px;margin-top:-8px;margin-bottom:10px;font-weight:500;">{signup_errs["sa"]}</div>', unsafe_allow_html=True)

            st.markdown("<br>", unsafe_allow_html=True)

            # Show specific error message(s)
            if signup_errs:
                if len(signup_errs) == 1:
                    st.error(list(signup_errs.values())[0])
                else:
                    err_items = "\n".join([f"- {msg}" for msg in signup_errs.values()])
                    st.error(f"Please fill in all required fields:\n{err_items}")

            if st.button("Create Account", use_container_width=True):
                new_errors = {}
                if not uname:
                    new_errors["uname"] = "Please enter your username."
                if not email:
                    new_errors["email"] = "Please enter your email address."
                if not pwd:
                    new_errors["pwd"] = "Please enter your password."
                if not confirm_pwd:
                    new_errors["confirm_pwd"] = "Please confirm your password."
                if not sq or sq == "Select a security question...":
                    new_errors["sq"] = "Please select a security question."
                if not sa:
                    new_errors["sa"] = "Please enter your security answer."

                if new_errors:
                    st.session_state.signup_errors = new_errors
                    st.rerun()

                st.session_state.signup_errors = {}

                if len(uname) < 3:
                    st.error("Username must be at least 3 characters long.")
                else:
                    is_valid_email, email_msg = validate_email(email)
                    if not is_valid_email:
                        st.error(f"Invalid Email: {email_msg}")
                    else:
                        is_valid_pwd, pwd_msg = validate_password(pwd)
                        if not is_valid_pwd:
                            st.error(f"Password Error: {pwd_msg}")
                        elif pwd != confirm_pwd:
                            st.error("Passwords do not match.")
                        elif email == ADMIN_EMAIL.lower() or uname.lower() == "administrator":
                            st.error("This username or email is reserved.")
                        else:
                            try:
                                now_str = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
                                new_hash = hash_txt(pwd)
                                with get_db() as c:
                                    c.execute(
                                        "INSERT INTO users (username, email, password_hash, security_question, security_answer_hash, role, created_at) "
                                        "VALUES (?, ?, ?, ?, ?, ?, ?)",
                                        (uname, email, new_hash, sq, hash_txt(sa.lower()), "user", now_str)
                                    )
                                    record_password_history(c, email, new_hash)
                                st.session_state.signup_errors = {}
                                st.success("Account created successfully. Please sign in with your credentials.")
                                time.sleep(1.5)
                                navigate("Login")
                            except sqlite3.IntegrityError:
                                st.error("Username or Email address already registered.")

            st.markdown("<br>", unsafe_allow_html=True)
            if st.button("Back to Sign In", use_container_width=True):
                st.session_state.signup_errors = {}
                navigate("Login")

        # FORGOT PASSWORD

        elif st.session_state.page == "Forgot":
            auth_header("Reset your password", "Choose your verification method")

            # Selection Step: Enter Username or Email and select recovery method

            if not st.session_state.reset_mode:
                recovery_id = st.text_input("Registered Username or Email address", placeholder="you@infosys.com or username").strip()
                st.markdown("<br>", unsafe_allow_html=True)

                col_sq, col_otp = st.columns(2)
                if col_sq.button("Via Security Question", use_container_width=True):
                    if not recovery_id:
                        st.error("Please enter your registered username or email.")
                    else:
                        with get_db() as c:
                            r = c.execute(
                                "SELECT email, security_question FROM users WHERE email=? OR username=?",
                                (recovery_id.lower(), recovery_id)
                            ).fetchone()
                        if r:
                            st.session_state.reset_email = r[0]
                            st.session_state.sq_p = r[1]
                            st.session_state.reset_mode = "sq"
                            st.rerun()
                        else:
                            st.error("Invalid credentials. Please verify your username or email.")

                if col_otp.button("Via Email OTP", use_container_width=True):
                    if not recovery_id:
                        st.error("Please enter your registered username or email address.")
                    else:
                        with get_db() as c:
                            r = c.execute(
                                "SELECT email FROM users WHERE email=? OR username=?",
                                (recovery_id.lower(), recovery_id)
                            ).fetchone()
                        if r:
                            st.session_state.reset_email = r[0]
                            st.session_state.reset_mode = "otp"
                            st.session_state.otp_stage = "email"
                            st.rerun()
                        else:
                            st.error("Invalid credentials. Please verify your username or email.")

            # Security Question Recovery
            elif st.session_state.reset_mode == "sq":
                st.info(f"Security Question: {st.session_state.sq_p}")
                ans = st.text_input("Your answer", placeholder="Security answer").strip()
                npw = st.text_input("New password", type="password", placeholder="Min. 8 chars (upper, lower, digit, symbol)")
                confirm_npw = st.text_input("Confirm new password", type="password", placeholder="Re-enter new password")
                st.markdown("<br>", unsafe_allow_html=True)

                if st.button("Reset Password", use_container_width=True):
                    if not ans:
                        st.error("Please provide the answer to your security question.")
                    else:
                        is_valid_pwd, pwd_msg = validate_password(npw)
                        if not is_valid_pwd:
                            st.error(f"Password Error: {pwd_msg}")
                        elif npw != confirm_npw:
                            st.error("Passwords do not match.")
                        else:
                            with get_db() as c:
                                r = c.execute(
                                    "SELECT security_answer_hash FROM users WHERE email=?",
                                    (st.session_state.reset_email,)
                                ).fetchone()
                                if r and check_txt(ans.lower(), r[0]):
                                    if check_password_reused(c, st.session_state.reset_email, npw):
                                        st.error("For security, you cannot reuse your current or recent password.")
                                    else:
                                        new_h = hash_txt(npw)
                                        c.execute("UPDATE users SET password_hash=? WHERE email=?", (new_h, st.session_state.reset_email))
                                        record_password_history(c, st.session_state.reset_email, new_h)
                                        clear_reset_state()
                                        st.success("Password updated successfully. Please sign in with your new password.")
                                        time.sleep(1.5)
                                        navigate("Login")
                                else:
                                    st.error("Incorrect security answer.")

            # Email OTP Recovery
            elif st.session_state.reset_mode == "otp":
                # Request OTP with Cooldown Protection
                if st.session_state.otp_stage == "email":
                    st.info("A 6-digit verification code will be sent to the registered email address on file.")
                    can_send, wait_sec = can_request_otp(st.session_state.get("otp_last_sent_at", 0), cooldown_seconds=60)

                    if not can_send:
                        st.warning(f"Please wait {wait_sec} seconds before requesting a new code.")

                    if st.button("Send Verification Code", use_container_width=True, disabled=not can_send):
                        if not EMAIL_PASSWORD:
                            st.error("Gmail App Password not found. Please configure EMAIL_PASSWORD in Colab Secrets.")
                        else:
                            otp = generate_otp()
                            with st.spinner("Sending verification code..."):
                                ok, msg = send_professional_email(SENDER_EMAIL, st.session_state.reset_email, otp, EMAIL_PASSWORD, OTP_EXPIRY_MINUTES)
                            if ok:
                                st.session_state.otp_token = make_otp_token(st.session_state.reset_email, otp, JWT_SECRET, OTP_EXPIRY_MINUTES)
                                st.session_state.otp_stage = "otp"
                                st.session_state.otp_attempts = 0
                                st.session_state.otp_last_sent_at = time.time()
                                st.success("Verification code sent to your registered email address. Please check your inbox.")
                                time.sleep(1)
                                st.rerun()
                            else:
                                st.error(f"Failed to send email: {msg}")

                # Enter & Verify OTP with Max Attempt Guard (3 attempts)
                elif st.session_state.otp_stage == "otp":
                    st.info(f"Verification code sent. Valid for {OTP_EXPIRY_MINUTES} minutes.")
                    otp_input = st.text_input("Enter 6-Digit Verification Code", placeholder="e.g. 849201", max_chars=6).strip()
                    st.markdown("<br>", unsafe_allow_html=True)

                    c1, c2 = st.columns([1.2, 1])
                    if c1.button("Verify Code", use_container_width=True):
                        if not otp_input or len(otp_input) != 6:
                            st.error("Please enter a valid 6-digit numerical code.")
                        else:
                            ok, msg = verify_otp_token(st.session_state.otp_token, otp_input, st.session_state.reset_email, JWT_SECRET, OTP_EXPIRY_MINUTES)
                            if ok:
                                st.session_state.otp_verified = True
                                st.session_state.otp_token = None  # Invalidate OTP token immediately after verification
                                st.session_state.otp_stage = "reset"
                                st.session_state.otp_attempts = 0
                                st.success("Code verified successfully.")
                                time.sleep(1)
                                st.rerun()
                            else:
                                st.session_state.otp_attempts = st.session_state.get("otp_attempts", 0) + 1
                                attempts_left = 3 - st.session_state.otp_attempts
                                if attempts_left > 0:
                                    st.error(f"{msg} ({attempts_left} attempt{'s' if attempts_left > 1 else ''} remaining)")
                                else:
                                    st.error("Maximum OTP attempts reached (3/3). This code has been invalidated. Please request a new code.")
                                    st.session_state.otp_token = None
                                    st.session_state.otp_stage = "email"
                                    st.session_state.otp_attempts = 0
                                    time.sleep(2)
                                    st.rerun()

                    if c2.button("Resend / Change", use_container_width=True):
                        st.session_state.otp_stage = "email"
                        st.session_state.otp_token = None
                        st.session_state.otp_attempts = 0
                        st.rerun()

                # Set New Password after verification
                elif st.session_state.otp_stage == "reset":
                    if not st.session_state.get("otp_verified"):
                        st.error("Unauthorized access. Please verify your OTP code first.")
                        st.session_state.otp_stage = "email"
                        st.rerun()

                    npw = st.text_input("New Password", type="password", placeholder="Min. 8 chars (upper, lower, digit, symbol)")
                    confirm_npw = st.text_input("Confirm New Password", type="password", placeholder="Re-enter new password")
                    st.markdown("<br>", unsafe_allow_html=True)

                    c1, c2 = st.columns([1.2, 1])
                    if c1.button("Update Password", use_container_width=True):
                        is_valid_pwd, pwd_msg = validate_password(npw)
                        if not is_valid_pwd:
                            st.error(f"Password Error: {pwd_msg}")
                        elif npw != confirm_npw:
                            st.error("Passwords do not match.")
                        else:
                            with get_db() as c:
                                if check_password_reused(c, st.session_state.reset_email, npw):
                                    st.error("For security, you cannot reuse your current or recent password. Please choose a new password.")
                                else:
                                    new_h = hash_txt(npw)
                                    c.execute("UPDATE users SET password_hash=? WHERE email=?", (new_h, st.session_state.reset_email))
                                    record_password_history(c, st.session_state.reset_email, new_h)
                                    clear_reset_state()
                                    st.success("Password updated successfully. Please log in with your new password.")
                                    time.sleep(1.5)
                                    navigate("Login")
                    if c2.button("Cancel", use_container_width=True):
                        clear_reset_state()
                        navigate("Login")

            st.markdown("<br>", unsafe_allow_html=True)
            if st.button("Cancel Recovery", use_container_width=True):
                clear_reset_state()
                navigate("Login")


# PROTECTED DASHBOARDS

else:
    # Server-Side JWT Validation
    payload = verify_jwt(st.session_state.token)
    if not payload:
        st.session_state.token = None
        st.session_state.page = "Login"
        st.rerun()

    auth_email = payload.get("email")

    # Server-Side Role Verification from Database
    with get_db() as c:
        user_row = c.execute("SELECT username, role FROM users WHERE email=?", (auth_email,)).fetchone()

    if not user_row:
        st.session_state.token = None
        st.session_state.page = "Login"
        st.rerun()

    uname, role = user_row[0], user_row[1] if user_row[1] else "user"
    is_admin = (role == "admin" or auth_email.lower() == ADMIN_EMAIL.lower())

    # SIDEBAR NAVIGATION

    with st.sidebar:
        st.markdown(f"""
        <div style="padding:16px 8px;text-align:center;">
            <div style="font-weight:700;font-size:16px;color:{COLORS['text_heading']};">FreightQuote AI</div>
            <div style="font-size:12px;color:{COLORS['text_muted']};margin-top:2px;">
                {"Admin Control Panel" if is_admin else "User Portal"}
            </div>
        </div>
        <hr style="border:none;border-top:1px solid {COLORS['border']};margin:8px 0 16px 0;">
        """, unsafe_allow_html=True)

        menu = option_menu(
            None, ["Dashboard", "Logout"],
            icons=["house", "box-arrow-right"],
            styles={
                "container": {"background-color": COLORS['bg_sidebar'], "padding": "0"},
                "nav-link": {"font-size": "14px", "color": COLORS['text_main'], "border-radius": "6px"},
                "nav-link-selected": {"background-color": COLORS['accent_hover'], "color": COLORS['text_heading'], "font-weight": "600"}
            }
        )

        if menu == "Logout":
            st.session_state.token = None
            st.session_state.page = "Login"
            clear_reset_state()
            st.rerun()

    # ADMIN DASHBOARD

    if is_admin:
        # Header banner identifying administrator
        st.markdown(f"""
        <div style="background:{COLORS['bg_card']};border:1px solid {COLORS['border']};border-radius:10px;padding:20px 24px;margin-bottom:24px;display:flex;justify-content:space-between;align-items:center;">
            <div>
                <h2 style="margin:0;font-size:20px;color:{COLORS['text_heading']};">FreightQuote AI — Admin Dashboard</h2>
                <div style="color:{COLORS['text_muted']};font-size:13px;margin-top:4px;">Logged in as: {uname}</div>
            </div>
            <div style="background:{COLORS['accent_hover']};border:1px solid {COLORS['accent']};padding:6px 14px;border-radius:6px;font-size:13px;font-weight:600;color:{COLORS['text_heading']};">
                Admin
            </div>
        </div>
        """, unsafe_allow_html=True)

        # Registered-Users Count metric
        with get_db() as c:
            total_users = c.execute("SELECT COUNT(*) FROM users").fetchone()[0]

        st.markdown(f"""
        <div class="pn-card" style="margin-bottom:20px;padding:16px 20px;">
            <div style="font-size:13px;color:{COLORS['text_muted']};font-weight:500;">Total Registered Users</div>
            <div style="font-size:24px;font-weight:700;color:{COLORS['text_heading']};margin-top:2px;">{total_users}</div>
        </div>
        """, unsafe_allow_html=True)


        st.markdown(f"""
        <div style="margin-bottom:12px;">
            <h3 style="font-size:16px;margin:0;color:{COLORS['text_heading']};">Registered User Accounts</h3>
            <div style="color:{COLORS['text_muted']};font-size:13px;margin-top:2px;">View and manage registered accounts securely. Sensitive information is protected.</div>
        </div>
        """, unsafe_allow_html=True)

        with get_db() as c:
            users_list = c.execute("SELECT id, username, email, role, created_at FROM users ORDER BY id ASC").fetchall()

        table_data = []
        for u in users_list:
            table_data.append({
                "User ID": u[0],
                "Username": u[1],
                "Email Address": u[2],
                "Role": u[3] if u[3] else "user",
                "Registered At": u[4] if u[4] else "N/A"
            })

        st.dataframe(table_data, use_container_width=True)


    # REGULAR USER DASHBOARD

    else:
        # User welcome banner
        st.markdown(f"""
        <div style="background:{COLORS['bg_card']};border:1px solid {COLORS['border']};border-radius:10px;padding:20px 24px;margin-bottom:24px;display:flex;justify-content:space-between;align-items:center;">
            <div>
                <h2 style="margin:0;font-size:20px;color:{COLORS['text_heading']};">Welcome, {uname}</h2>
                <div style="color:{COLORS['text_muted']};font-size:13px;margin-top:4px;">User Account Overview</div>
            </div>
            <div style="background:{COLORS['accent_hover']};border:1px solid {COLORS['accent']};padding:6px 14px;border-radius:6px;font-size:13px;font-weight:600;color:{COLORS['text_heading']};">
                Active Session
            </div>
        </div>
        """, unsafe_allow_html=True)

        # Account details summary card
        st.markdown(f"""
        <div class="pn-card">
            <h3 style="font-size:16px;margin-top:0;margin-bottom:16px;color:{COLORS['text_heading']};">Account Information</h3>
            <div class="account-row">
                <span class="account-label">Username</span>
                <span class="account-value">{uname}</span>
            </div>
            <div class="account-row">
                <span class="account-label">Email Address</span>
                <span class="account-value">{auth_email}</span>
            </div>
            <div class="account-row">
                <span class="account-label">Account Role</span>
                <span class="account-value">{role.capitalize()}</span>
            </div>
            <div class="account-row" style="border-bottom:none;">
                <span class="account-label">Authentication Status</span>
                <span class="account-value" style="color:#059669;">Verified</span>
            </div>
        </div>
        """, unsafe_allow_html=True)


In [ ]:
import os
import time
import subprocess
from pyngrok import ngrok

# 1. Safely retrieve ngrok authentication token from Colab Secrets
try:
    from google.colab import userdata
    NGROK_TOKEN = userdata.get("NGROK_AUTHTOKEN")
    colab_env = True
except Exception:
    NGROK_TOKEN = os.getenv("NGROK_AUTHTOKEN")
    userdata = None
    colab_env = False

if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)
else:
    print(" NGROK_AUTHTOKEN not found in Colab Secrets. Add 'NGROK_AUTHTOKEN' to your Colab Secrets tab.")

# 2. Kill any existing ngrok tunnels or previous streamlit sessions cleanly
ngrok.kill()
!pkill -f streamlit
time.sleep(2)

# 3. Propagate Colab Secrets safely to the Streamlit subprocess environment
sub_env = os.environ.copy()
for secret_key in ["EMAIL_PASSWORD", "JWT_SECRET", "EMAIL_ID", "SENDER_EMAIL", "ADMIN_PASSWORD", "ADMIN_EMAIL"]:
    try:
        val = userdata.get(secret_key) if userdata else os.getenv(secret_key)
        if val:
            sub_env[secret_key] = val
    except Exception:
        pass

# 4. Start Streamlit in the background on port 8501
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501"],
    env=sub_env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(3)

# 5. Connect and expose public ngrok tunnel
try:
    public_url = ngrok.connect(8501).public_url

    print(f"FreightQuote AI Live URL: {public_url}")
    print("\n")
    print("App is running! Press [Ctrl + C] or the Colab Stop button to shut down.")

    while True:
        time.sleep(1)

except KeyboardInterrupt:
    print("\n")
    print("Received Ctrl+C / Stop signal. Shutting down...")
    ngrok.kill()
    process.terminate()
    !pkill -f streamlit
    print("\n")
    print("Ngrok tunnel closed and Streamlit server stopped gracefully.")
